In [1]:
# %% [markdown]
# # 06_statistical_reanalysis.ipynb
# Final statistical comparison of v1 (Semantic Layer, original
# architecture) vs v2 (Semantic Layer, true information-hiding
# architecture) on the Phase 3 credit rating benchmark, using
# question-level aggregation throughout -- directly addressing:
#
#   R1 Concern #2 (independence violation): the original analysis
#   treated 5 iterations per question as 5 independent observations.
#   This notebook aggregates to ONE accuracy value per question per
#   method before running any significance test, matching the
#   approach McNemar's test already used correctly.
#
#   R1 Concern #3 (equivalence via non-significance): rather than
#   reading a non-significant p-value as "equivalent," this notebook
#   runs a formal TOST-style equivalence test against pre-specified
#   margins, so any equivalence claim in the manuscript is backed by
#   an actual equivalence test rather than a failure to reject a null.
#
# Also produces the corrected Table 3 (exposure) and a complete
# failure taxonomy (R1 Concern #7) for v2.
#
# Requires: results_phase3_v2.csv (v1, from the ORIGINAL
# credit_rating_benchmark_experiment.ipynb) and
# results_phase3_v2_arch.csv (v2, from 04_full_experiment.ipynb) both
# present in this directory.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

df_v1 = pd.read_csv('results_phase3_v2.csv')
df_v2 = pd.read_csv('results_phase3_v2_arch.csv')

print(f"v1 (original Semantic Layer): {len(df_v1)} records, "
      f"{df_v1['question_id'].nunique()} questions, "
      f"{df_v1['iteration'].nunique()} iterations")
print(f"v2 (true info-hiding):        {len(df_v2)} records, "
      f"{df_v2['question_id'].nunique()} questions, "
      f"{df_v2['iteration'].nunique()} iterations")

# %% [markdown]
# ## 1. Question-level accuracy aggregation
#     (mirrors the aggregation McNemar's test already used correctly
#     in the original statistical_validation.ipynb -- extended here
#     to ALL tests, not just McNemar)

# %%
def question_level_accuracy(df):
    return df.groupby('question_id')['is_correct'].mean()

q_v1 = question_level_accuracy(df_v1)
q_v2 = question_level_accuracy(df_v2)

print(f"v1 question-level accuracy: n={len(q_v1)} questions, "
      f"mean={q_v1.mean():.4f}, sd={q_v1.std():.4f}")
print(f"v2 question-level accuracy: n={len(q_v2)} questions, "
      f"mean={q_v2.mean():.4f}, sd={q_v2.std():.4f}")

# %% [markdown]
# ## 2. Paired comparison: v1 vs v2 accuracy
#     Since both used the same 60-question set, this is a natural
#     paired design at the question level.

# %%
print("=" * 70)
print("PAIRED COMPARISON: v1 vs v2 (question-level, n=60 pairs)")
print("=" * 70)

# align on question_id to guarantee correct pairing
common_qids = sorted(set(q_v1.index) & set(q_v2.index))
if len(common_qids) != 60:
    print(f"!! WARNING: only {len(common_qids)} question_ids common to both "
          f"files (expected 60). Check both files cover the same benchmark.")

v1_aligned = q_v1.loc[common_qids].values
v2_aligned = q_v2.loc[common_qids].values
diff = v2_aligned - v1_aligned

print(f"\nn = {len(common_qids)} question pairs")
print(f"v1 mean question-level accuracy: {v1_aligned.mean():.4f}")
print(f"v2 mean question-level accuracy: {v2_aligned.mean():.4f}")
print(f"Mean difference (v2 - v1): {diff.mean():+.4f}")

t_stat, t_p = stats.ttest_rel(v2_aligned, v1_aligned)
try:
    w_stat, w_p = stats.wilcoxon(v2_aligned, v1_aligned)
except ValueError as e:
    w_stat, w_p = np.nan, np.nan
    print(f"(Wilcoxon skipped: {e})")

print(f"\nPaired t-test:        t={t_stat:.4f}, p={t_p:.4g}")
print(f"Wilcoxon signed-rank: W={w_stat:.4f}, p={w_p:.4g}")
print(f"\nQuestions where v2 > v1: {(diff > 0).sum()}, "
      f"v1 > v2: {(diff < 0).sum()}, tied: {(diff == 0).sum()}")

# %% [markdown]
# ## 3. Question-level bootstrap 95% CI
#     (resamples QUESTIONS, not individual generations -- directly
#     addresses R1's explicit request: "Bootstrap at the question
#     level, not at the individual generation level.")

# %%
def question_level_bootstrap(values_a, values_b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(values_a)
    boot_diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_diffs[i] = values_b[idx].mean() - values_a[idx].mean()
    ci = np.percentile(boot_diffs, [2.5, 97.5])
    return ci, boot_diffs

print("=" * 70)
print("QUESTION-LEVEL BOOTSTRAP 95% CI (n_boot=5000, seed=42)")
print("=" * 70)

ci_diff, boot_diffs = question_level_bootstrap(v1_aligned, v2_aligned)
print(f"\n95% CI for (v2 - v1) mean accuracy difference: "
      f"[{ci_diff[0]:+.4f}, {ci_diff[1]:+.4f}]")
contains_zero = ci_diff[0] <= 0 <= ci_diff[1]
print(f"CI contains zero: {contains_zero}")

# %% [markdown]
# ## 4. TOST-style equivalence test
#     Per R1 Concern #3: rather than treating non-significance as
#     equivalence, this tests explicitly against pre-specified
#     margins. NOTE: the choice of margin should be justified in the
#     manuscript's methods section as practically meaningful for
#     credit-rating query use, not chosen post-hoc to produce a
#     desired result -- these candidate margins are shown for
#     transparency, not as a recommendation of which to report.

# %%
def tost_equivalence(diff_values, margin):
    n = len(diff_values)
    mean_diff = diff_values.mean()
    se = diff_values.std(ddof=1) / np.sqrt(n)

    t_lower = (mean_diff - (-margin)) / se
    t_upper = (mean_diff - margin) / se
    p_lower = 1 - stats.t.cdf(t_lower, df=n-1)
    p_upper = stats.t.cdf(t_upper, df=n-1)
    p_tost = max(p_lower, p_upper)

    t_crit = stats.t.ppf(0.95, df=n-1)
    ci_90 = (mean_diff - t_crit*se, mean_diff + t_crit*se)
    equivalent = (ci_90[0] > -margin) and (ci_90[1] < margin)
    return mean_diff, ci_90, p_tost, equivalent

print("=" * 70)
print("TOST-STYLE EQUIVALENCE TEST (v2 vs v1, various candidate margins)")
print("=" * 70)
print()
for margin in [0.05, 0.10, 0.15]:
    mean_diff, ci_90, p_tost, equivalent = tost_equivalence(diff, margin)
    verdict = "EQUIVALENT" if equivalent else "NOT equivalent"
    print(f"margin=±{margin:.2f} ({margin*100:.0f}pp): mean_diff={mean_diff:+.4f}, "
          f"90% CI=[{ci_90[0]:+.4f}, {ci_90[1]:+.4f}], "
          f"TOST p={p_tost:.4g}  -> {verdict}")

# %% [markdown]
# ## 5. Complete failure taxonomy for v2
#     (per R1 Concern #7: "Report a complete failure taxonomy:
#     correct executable, incorrect executable, execution failure,
#     timeout, and empty or degenerate result.")

# %%
print("\n" + "=" * 70)
print("COMPLETE FAILURE TAXONOMY -- v2")
print("=" * 70)

total_v2 = len(df_v2)
n_correct = df_v2['is_correct'].sum()
n_compile_err = df_v2['compile_error'].notna().sum()
n_exec_err_all = df_v2['execution_error'].notna().sum()
n_timeout = df_v2['execution_error'].str.contains('timeout', case=False, na=False).sum()
n_exec_err_real = n_exec_err_all - n_timeout
n_wrong = total_v2 - n_correct - n_compile_err - n_exec_err_all

print(f"\n  {'Category':<35} {'n':>6} {'%':>8}")
print(f"  {'-'*35} {'-'*6} {'-'*8}")
print(f"  {'Correct (executable, matches gold)':<35} {n_correct:>6} {n_correct/total_v2:>7.1%}")
print(f"  {'Compile failure (concept-SQL rejected)':<35} {n_compile_err:>6} {n_compile_err/total_v2:>7.1%}")
print(f"  {'Execution failure (invalid SQL)':<35} {n_exec_err_real:>6} {n_exec_err_real/total_v2:>7.1%}")
print(f"  {'Timeout':<35} {n_timeout:>6} {n_timeout/total_v2:>7.1%}")
print(f"  {'Wrong result (executable, mismatched)':<35} {n_wrong:>6} {n_wrong/total_v2:>7.1%}")
print(f"  {'-'*35} {'-'*6} {'-'*8}")
print(f"  {'Total':<35} {total_v2:>6} {'100.0%':>8}")

# compile-error subtype breakdown, since this is the largest single
# non-correct category and its composition matters for the manuscript
def classify_compile_error(err):
    if pd.isna(err):
        return None
    if 'WHERE clause' in err:
        return 'measure-in-WHERE'
    elif 'MISMATCHED' in err:
        return 'join hint mismatched'
    elif 'does not correspond to any defined relationship' in err:
        return 'join hint unresolved'
    elif 'Excluded concept' in err:
        return 'excluded concept used'
    else:
        return 'other'

df_v2['compile_error_type'] = df_v2['compile_error'].apply(classify_compile_error)
print("\nCompile-error subtypes:")
print(df_v2['compile_error_type'].value_counts())

# %% [markdown]
# ## 6. Category-level comparison: v1 vs v2

# %%
print("\n" + "=" * 70)
print("CATEGORY-LEVEL ACCURACY: v1 vs v2")
print("=" * 70)

cat_v1 = df_v1.groupby('category')['is_correct'].mean()
cat_v2 = df_v2.groupby('category')['is_correct'].mean()
cat_compare = pd.DataFrame({'v1': cat_v1, 'v2': cat_v2})
cat_compare['diff'] = cat_compare['v2'] - cat_compare['v1']
print(cat_compare)

# %% [markdown]
# ## 7. Difficulty-level comparison: v1 vs v2 (Jagged Frontier check)

# %%
print("\n" + "=" * 70)
print("DIFFICULTY-LEVEL ACCURACY: v1 vs v2")
print("=" * 70)

diff_v1 = df_v1.groupby('difficulty')['is_correct'].mean().reindex(['simple','moderate','challenging'])
diff_v2 = df_v2.groupby('difficulty')['is_correct'].mean().reindex(['simple','moderate','challenging'])
diff_compare = pd.DataFrame({'v1': diff_v1, 'v2': diff_v2})
diff_compare['diff'] = diff_compare['v2'] - diff_compare['v1']
print(diff_compare)

# %% [markdown]
# ## 8. Exposure comparison summary (final, for manuscript Table 3)

# %%
print("\n" + "=" * 70)
print("FINAL EXPOSURE COMPARISON (manuscript Table 3 replacement)")
print("=" * 70)
print("""
  Metric                              Text-to-SQL   SL v1        SL v2
  ---------------------------------   -----------   ----------   ----------
  Sensitive columns exposed (of 15)         15          15 (0%)      0 (100%)
  Internal codes exposed (of 9)              0           0            0
  Verified across                            --      60 questions  60 questions
                                                       (exhaustive)  (exhaustive)

  NOTE: SL v1's originally-reported "100% reduction" was found NOT
  to hold under exhaustive verification of the actual API payload
  (concept-to-SQL mapping table + table reference + few-shot
  examples all contain raw column identifiers). SL v2 is the
  corrected true information-hiding architecture: the LLM-facing
  prompt contains ONLY concept names/descriptions; compilation to
  real SQL happens locally, after the API call returns. This is the
  architecture that achieves the 100% reduction originally claimed.
""")

# %% [markdown]
# ## 9. Final summary table for the manuscript

# %%
print("=" * 70)
print("FINAL SUMMARY: v1 vs v2 HEADLINE COMPARISON")
print("=" * 70)
print(f"""
  Metric                          Text-to-SQL    SL v1          SL v2
  ------------------------------  -----------    -----------    -----------
  Overall accuracy                 39.0%          38.3%          {v2_aligned.mean():.1%}
  Sensitive columns exposed        15/15          15/15 (0%)     0/15 (100%)
  Question-level paired test       --             --             t={t_stat:.2f}, p={t_p:.4g}
  vs v1 (v2 comparison only)

  KEY FINDING: v2 achieves the 100% schema-level exposure reduction
  originally claimed for the Semantic Layer architecture, at a cost
  of {abs(v2_aligned.mean() - v1_aligned.mean()):.1%} accuracy relative to v1 (which does NOT achieve
  the claimed exposure reduction). This is the central security-
  efficiency tradeoff the revised manuscript should report.
""")

print("Done. Use these tables directly for the manuscript's revised")
print("Table 3 (exposure), Table 4 (statistical summary), and the new")
print("failure taxonomy table addressing R1 Concern #7.")

SCRIPT VERSION: 2026-07-15-v1
v1 (original Semantic Layer): 600 records, 60 questions, 5 iterations
v2 (true info-hiding):        300 records, 60 questions, 5 iterations
v1 question-level accuracy: n=60 questions, mean=0.3867, sd=0.4168
v2 question-level accuracy: n=60 questions, mean=0.3867, sd=0.4685
PAIRED COMPARISON: v1 vs v2 (question-level, n=60 pairs)

n = 60 question pairs
v1 mean question-level accuracy: 0.3867
v2 mean question-level accuracy: 0.3867
Mean difference (v2 - v1): -0.0000

Paired t-test:        t=-0.0000, p=1
Wilcoxon signed-rank: W=214.5000, p=0.9482

Questions where v2 > v1: 15, v1 > v2: 14, tied: 31
QUESTION-LEVEL BOOTSTRAP 95% CI (n_boot=5000, seed=42)

95% CI for (v2 - v1) mean accuracy difference: [-0.1033, +0.0983]
CI contains zero: True
TOST-STYLE EQUIVALENCE TEST (v2 vs v1, various candidate margins)

margin=±0.05 (5pp): mean_diff=-0.0000, 90% CI=[-0.0869, +0.0869], TOST p=0.1702  -> NOT equivalent
margin=±0.10 (10pp): mean_diff=-0.0000, 90% CI=[-0.0869, 